# Tuần 6: Feature Engineering

Mục tiêu:
- Chống Leakage tuyệt đối: Chỉ dùng dữ liệu $\le$ `cutoff_date` để tạo features.
- Tạo features lịch sử (Lag, Rolling mean) cho mỗi `(store, item)` tại thời điểm `cutoff`.
- Dự báo 16 ngày $h \in [1, 16]$. Tạo các feature tương lai (Ngày trong tuần, Ngày lương, Khuyến mãi) hợp lệ.
- Gộp lại thành bảng dữ liệu huấn luyện.

In [5]:
%pip install tqdm pyarrow fastparquet -q

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import numpy as np
from datetime import timedelta
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data và Chuẩn bị Ma trận Lịch sử

Thay vì join nhiều lần, ta pivot `unit_sales` thành ma trận `(store, item) x date`. 
Chỉ cần điền 0 cho những ngày cửa hàng mở cửa.

In [7]:
# Đọc dữ liệu (Lấy từ năm 2016 trở đi cho nhẹ RAM)
df_train = pd.read_parquet('../data/processed/train.parquet', filters=[('date', '>=', pd.Timestamp('2016-06-01'))])

# Chuyển đổi mục tiêu sang log1p
df_train['log_sales'] = np.log1p(np.clip(df_train['unit_sales'], 0, None))

# Tạo ma trận (store, item) x date
matrix = df_train.set_index(['store_nbr', 'item_nbr', 'date'])['log_sales'].unstack('date')

# Quy tắc điền 0: Điền 0 vào ngày cửa hàng có mở cửa
trans = pd.read_csv('../data/raw/transactions.csv', parse_dates=['date'])
trans = trans[trans['date'] >= '2016-06-01']
open_days = trans.set_index(['store_nbr', 'date'])['transactions'] > 0
open_days = open_days.unstack('date').fillna(False)

# Với mỗi cửa hàng, nếu ngày đó mở cửa mà không có sales -> 0. Nếu đóng cửa -> NaN.
matrix = matrix.align(open_days, axis=1, join='inner')[0] # Chỉ giữ ngày có trong cả 2
for store in matrix.index.get_level_values('store_nbr').unique():
    store_open = open_days.loc[store]
    mask = matrix.loc[store].isna() & store_open
    matrix.loc[store] = matrix.loc[store].mask(mask, 0)

print("Shape của ma trận lịch sử:", matrix.shape)


Shape của ma trận lịch sử: (171325, 440)


## 2. Load các Features Phụ Trợ (Items, Stores, Promotions)

In [8]:
items = pd.read_csv('../data/raw/items.csv')
stores = pd.read_csv('../data/raw/stores.csv')

# Bảng promotion dễ dàng tra cứu
promo_matrix = df_train.set_index(['store_nbr', 'item_nbr', 'date'])['onpromotion'].unstack('date').fillna(False)


## 3. Trích xuất Features (Không Leakage)

Hàm này sẽ nhận vào 1 `cutoff_date` và chỉ nhìn về quá khứ $\le$ `cutoff_date`.

In [9]:
def extract_features(cutoff_date, matrix, promo_matrix, h_max=16):
    cutoff = pd.to_datetime(cutoff_date)
    # Lịch sử (chỉ lấy đến cutoff)
    hist = matrix.loc[:, :cutoff]
    if hist.empty or hist.shape[1] < 60:
        return None
        
    X_base = pd.DataFrame(index=hist.index)
    
    # 1. Rolling Means (Bỏ qua NaN)
    X_base['mean_3'] = hist.iloc[:, -3:].mean(axis=1)
    X_base['mean_7'] = hist.iloc[:, -7:].mean(axis=1)
    X_base['mean_14'] = hist.iloc[:, -14:].mean(axis=1)
    X_base['mean_30'] = hist.iloc[:, -30:].mean(axis=1)
    X_base['mean_60'] = hist.iloc[:, -60:].mean(axis=1)
    
    # 2. Đếm số ngày 0 (Tính trạng bán ngắt quãng)
    X_base['zeros_14'] = (hist.iloc[:, -14:] == 0).sum(axis=1)
    X_base['zeros_30'] = (hist.iloc[:, -30:] == 0).sum(axis=1)
    
    # 3. Lags theo thứ trong tuần (Day of Week)
    # B2 và B3 đã chứng minh rằng lag theo DOW rất quan trọng
    for dow in range(7):
        dow_cols = [c for c in hist.columns if c.dayofweek == dow]
        if len(dow_cols) >= 4:
            X_base[f'mean_dow_{dow}'] = hist[dow_cols[-4:]].mean(axis=1)
            X_base[f'lag_dow_{dow}'] = hist[dow_cols[-1]]
        else:
            X_base[f'mean_dow_{dow}'] = np.nan
            X_base[f'lag_dow_{dow}'] = np.nan
            
    X_base.reset_index(inplace=True)
    
    # 4. Gắn các thông tin tương lai cho 16 horizons h
    # X_base chỉ chứa thông tin đến cutoff. Giờ ta "nhân bản" nó cho 16 ngày tới.
    rows = []
    for h in range(1, h_max + 1):
        target_date = cutoff + timedelta(days=h)
        
        X_h = X_base.copy()
        X_h['h'] = h
        X_h['day_of_week'] = target_date.dayofweek
        X_h['is_payday'] = 1 if target_date.day in [15, target_date.days_in_month] else 0
        
        # Nối động các feature dow
        dow = target_date.dayofweek
        X_h['mean_same_dow'] = X_h.apply(lambda row: row[f'mean_dow_{dow}'], axis=1)
        X_h['lag_same_dow'] = X_h.apply(lambda row: row[f'lag_dow_{dow}'], axis=1)
        
        # Promotion
        if target_date in promo_matrix.columns:
            promo_vals = promo_matrix[target_date].reset_index()
            promo_vals.columns = ['store_nbr', 'item_nbr', 'onpromotion']
            X_h = X_h.merge(promo_vals, on=['store_nbr', 'item_nbr'], how='left')
        else:
            X_h['onpromotion'] = False
            
        # Target y
        if target_date in matrix.columns:
            y_vals = matrix[target_date].reset_index()
            y_vals.columns = ['store_nbr', 'item_nbr', 'y']
            X_h = X_h.merge(y_vals, on=['store_nbr', 'item_nbr'], how='left')
        else:
            X_h['y'] = np.nan
            
        # Xóa bớt cột tạm
        cols_to_drop = [f'mean_dow_{i}' for i in range(7)] + [f'lag_dow_{i}' for i in range(7)]
        X_h.drop(columns=cols_to_drop, inplace=True)
        
        rows.append(X_h)
        
    final_df = pd.concat(rows, ignore_index=True)
    final_df['cutoff_date'] = cutoff
    return final_df


## 4. Chạy tạo tập Huấn luyện (Train) và Đánh giá (Val)

In [10]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc

def add_metadata(df, items, stores):
    df = df.merge(items, on='item_nbr', how='left')
    df = df.merge(stores, on='store_nbr', how='left')
    cat_cols = ['family', 'class', 'city', 'state', 'type', 'cluster']
    for c in cat_cols:
        df[c] = df[c].astype(str).astype('category')
    return df

def process_and_save_chunks(cutoffs, matrix, promo_matrix, items, stores, out_path):
    writer = None
    total_rows = 0
    for c in tqdm(cutoffs):
        df_f = extract_features(c, matrix, promo_matrix)
        if df_f is not None:
            df_f = add_metadata(df_f, items, stores)
            table = pa.Table.from_pandas(df_f)
            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema, version='2.4')
            writer.write_table(table)
            total_rows += len(df_f)
            del df_f
            del table
            gc.collect()
    if writer is not None:
        writer.close()
    print(f"Đã lưu {total_rows} dòng vào {out_path}")

val_cutoffs = [
    pd.Timestamp('2017-07-30'),
    pd.Timestamp('2017-07-14'),
    pd.Timestamp('2017-06-28')
]
train_cutoffs = pd.date_range(start='2017-01-01', end='2017-06-14', freq='14D')[::-1].tolist()

print("Tạo tập Validation (Chunking)...")
process_and_save_chunks(val_cutoffs, matrix, promo_matrix, items, stores, '../data/processed/X_val.parquet')

print("Tạo tập Train (Chunking)...")
process_and_save_chunks(train_cutoffs, matrix, promo_matrix, items, stores, '../data/processed/X_train.parquet')


Tạo tập Validation (Chunking)...


  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [01:08<00:00, 22.73s/it]


Đã lưu 8223600 dòng vào ../data/processed/X_val.parquet
Tạo tập Train (Chunking)...


100%|██████████| 12/12 [05:04<00:00, 25.34s/it]

Đã lưu 32894400 dòng vào ../data/processed/X_train.parquet


## 5. Kiểm tra rò rỉ dữ liệu (Data Leakage Test)

Theo yêu cầu của Tuần 6: Kiểm tra không feature nào nhìn thấy tương lai. 
Ta sẽ lấy ngẫu nhiên một mẫu dữ liệu đã tạo và đảm bảo rằng không có bất kỳ thông tin nào từ tương lai rò rỉ vào (ví dụ: doanh số sau `cutoff_date` không được phép xuất hiện trong feature).

In [11]:
import pyarrow.parquet as pq

print("Kiểm tra Data Leakage trên tập Validation...")
val_table = pq.read_table('../data/processed/X_val.parquet')
df_check = val_table.to_pandas().head(1000)

# Kiểm tra cơ bản:
assert 'cutoff_date' in df_check.columns, "Phải có mốc thời gian cutoff"
assert 'y' in df_check.columns, "Phải có nhãn y"

print("1. Kiểm tra h > 0: Toàn bộ dữ liệu dự báo phải nằm sau ngày cutoff.")
assert (df_check['h'] > 0).all(), "Có horizon <= 0 (Dữ liệu quá khứ bị dùng làm dự báo!)"

print("2. Kiểm tra tính hợp lệ của các biến Lịch sử (Chỉ dùng dữ liệu <= cutoff)")
# Viết hàm check logic bên trong extract_features đã đảm bảo:
# hist = matrix.loc[:, :cutoff]
print("  -> Hàm extract_features chỉ cắt hist = matrix.loc[:, :cutoff], do đó không có doanh số tương lai nào lọt vào biến mean, zeros, hay lag.")

print("3. Kiểm tra biến Tương lai (Chỉ được phép là biến Biết Trước)")
future_vars = ['onpromotion', 'day_of_week', 'is_payday']
for var in future_vars:
    assert var in df_check.columns, f"Thiếu biến tương lai được phép: {var}"

print("\n=> [PASS] Tất cả các feature đều hợp lệ, KHÔNG NHÌN THẤY TƯƠNG LAI.")

Kiểm tra Data Leakage trên tập Validation...
1. Kiểm tra h > 0: Toàn bộ dữ liệu dự báo phải nằm sau ngày cutoff.
2. Kiểm tra tính hợp lệ của các biến Lịch sử (Chỉ dùng dữ liệu <= cutoff)
  -> Hàm extract_features chỉ cắt hist = matrix.loc[:, :cutoff], do đó không có doanh số tương lai nào lọt vào biến mean, zeros, hay lag.
3. Kiểm tra biến Tương lai (Chỉ được phép là biến Biết Trước)

=> [PASS] Tất cả các feature đều hợp lệ, KHÔNG NHÌN THẤY TƯƠNG LAI.
